<a href="https://colab.research.google.com/github/donthaganiganesh96-tech/intelligent-budget-tracker/blob/main/My_First_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the necessary AI libraries
!pip install -q crewai[tools] langchain-groq python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.9/779.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os
from google.colab import userdata

# This pulls the keys you already saved in the "Secrets" (key icon) tab
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print("Keys loaded successfully from Secrets!")

Keys loaded successfully from Secrets!


In [23]:
!pip install litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 83.3 MB/s eta 0:00:00
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.1.1
    Uninstalling python-dotenv-1.1.1:
      Successfully uninstalled python-dotenv-1.1.1
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling pyda

In [7]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool

# 1. Set the LLM directly as a string for the Agent
# CrewAI will handle the connection using your environment variables
# Change this line:
llm_config = "groq/llama-3.3-70b-versatile"

# Or use the smaller, faster version:
# llm_config = "groq/llama-3.1-8b-instant"

# 2. Define the Agent
sentry = Agent(
    role='System Monitor',
    goal='Find bugs in a GitHub repo and research the fix.',
    backstory='You monitor logs and search StackOverflow for solutions.',
    tools=[SerperDevTool()],
    llm=llm_config,   # Use the string here
    verbose=True

)

developer = Agent(
    role='Senior Developer',
    goal='Write code to fix the bug and report the solution.',
    backstory='You turn research into clean, working Python code.',
    llm=llm_config,   # Use the string here too
    verbose=True,
    allow_delegation=False
)

# 3. Define the Task
task_fix = Task(
    description='Search for errors in the logic of a Python script, find a solution, and write the fix.',
    expected_output='A full code block containing the fix and an explanation.',
    agent=sentry
)

# 4. Assemble the Crew
my_crew = Crew(
    agents=[sentry, developer],
    tasks=[task_fix],
    process=Process.sequential
)

# 5. Execute
result = my_crew.kickoff()
print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: System Monitor                                                                                          │
│                                                                                                                 │
│  Task: Search for errors in the logic of a Python script, find a solution, and write the fix.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:crewai_tools.tools.serper_dev_tool.serper_dev_tool:Error making request to Serper API: Invalid leading whitespace, reserved character(s), or return character(s) in header value: '9902d410-3a12-11f1-b79f-d14bb5b83d86\r\nfrom google.colab import userdata\r\n\r\n# It\'s best to use Colab\'s "Secrets" (the key icon on the left)\r\n# But for now, you can just input them directly:\r\nos.environ["GROQ_API_KEY"] = "paste_your_groq_key_here"\r\nos.environ["SERPER_API_KEY"] = "paste_your_serper_search_key_here"\r\nos.environ["GITHUB_TOKEN"] = "paste_your_github_token_here"'


Tool search_the_internet_with_serper executed with result: Error executing tool: Invalid leading whitespace, reserved character(s), or return character(s) in header value: '9902d410-3a12-11f1-b79f-d14bb5b83d86\r\nfrom google.colab import userdata\r\n\r\n# It\...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: System Monitor                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Step-by-step analysis of the problem:                                                                        │
│  1. **Identifying the task**: The task requires searching for errors in the logic of a Python script, finding   │
│  a solution, and writing the fix.                                                                               │
│  2. **Recognizing the limitations**: The provided tool result does not contain a specific Python script to      │
│  analyze or a particular error to fix.                                                                          │
│  3. **Understanding the expected outcome**: The final answer should include a full code block containing the    │
│  fix and an explanation.                                                                                        │
│                                                                                                                 │
│  # Fixed solution:                                                                                              │
│  Since the problem lacks specifics about the Python script or the error in its logic, let's create a generic    │
│  example. Suppose we have a Python script that is supposed to calculate the average of a list of numbers but    │
│  contains a logical error.                                                                                      │
│                                                                                                                 │
│  ```python                                                                                                      │
│  # Original script with a logical error                                                                         │
│  def calculate_average(numbers):                                                                                │
│      sum_of_numbers = 0                                                                                         │
│      for number in numbers:                                                                                     │
│          sum_of_numbers += number                                                                               │
│      return sum_of_numbers  # Error: not dividing by the count of numbers                                       │
│                                                                                                                 │
│  # Example usage                                                                                                │
│  numbers = [1, 2, 3, 4, 5]                                                                                      │
│  average = calculate_average(numbers)                                                                           │
│  print("Average (with error):", average)                                                                        │
│                                                                                                                 │
│  # Fixed script                                                                                                 │
│  def calculate_average_fixed(numbers):                                                                          │
│      if len(numbers) == 0:                                                                                      │
│          return 0  # or any other appropriate value for

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

# Step-by-step analysis of the problem:
1. **Identifying the task**: The task requires searching for errors in the logic of a Python script, finding a solution, and writing the fix.
2. **Recognizing the limitations**: The provided tool result does not contain a specific Python script to analyze or a particular error to fix.
3. **Understanding the expected outcome**: The final answer should include a full code block containing the fix and an explanation.

# Fixed solution:
Since the problem lacks specifics about the Python script or the error in its logic, let's create a generic example. Suppose we have a Python script that is supposed to calculate the average of a list of numbers but contains a logical error.

```python
# Original script with a logical error
def calculate_average(numbers):
    sum_of_numbers = 0
    for number in numbers:
        sum_of_numbers += number
    return sum_of_numbers  # Error: not dividing by the count of numbers

# Example usage
numbers = [1, 2, 3, 4, 5]


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯